# 7. Predictive Analysis (Classification)
Four models are tuned with `GridSearchCV`, compared on the test set and the best one is explained with a confusion matrix.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix
data = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv")
X = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv")
Y = data['Class'].to_numpy()

In [ ]:
def plot_confusion_matrix(y, y_predict, title):
    cm = confusion_matrix(y, y_predict)
    ax = plt.subplot()
    sns.heatmap(cm, annot=True, ax=ax, fmt='d')
    ax.set_xlabel('Predicted labels'); ax.set_ylabel('True labels'); ax.set_title(title)
    ax.xaxis.set_ticklabels(['did not land', 'land']); ax.yaxis.set_ticklabels(['did not land', 'landed'])
    plt.show()

## Standardize and split (80/20)

In [ ]:
X = preprocessing.StandardScaler().fit(X).transform(X)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)
print('test samples:', Y_test.shape[0])

## Train and tune the models

In [ ]:
models = {
 'Logistic Regression': (LogisticRegression(), {'C': [0.01, 0.1, 1], 'penalty': ['l2'], 'solver': ['lbfgs']}),
 'SVM': (SVC(), {'kernel': ('linear', 'rbf', 'poly', 'sigmoid'), 'C': np.logspace(-3, 3, 5), 'gamma': np.logspace(-3, 3, 5)}),
 'Decision Tree': (DecisionTreeClassifier(), {'criterion': ['gini', 'entropy'], 'splitter': ['best', 'random'],
                   'max_depth': [2*n for n in range(1, 10)], 'max_features': ['sqrt', 'log2'],
                   'min_samples_leaf': [1, 2, 4], 'min_samples_split': [2, 5, 10]}),
 'KNN': (KNeighborsClassifier(), {'n_neighbors': list(range(1, 11)), 'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'], 'p': [1, 2]}),
}
fitted, results = {}, []
for name, (est, grid) in models.items():
    gs = GridSearchCV(est, grid, cv=10).fit(X_train, Y_train)
    fitted[name] = gs
    results.append({'Model': name, 'Best CV accuracy': gs.best_score_, 'Test accuracy': gs.score(X_test, Y_test)})
    print(name, gs.best_params_)
results_df = pd.DataFrame(results).set_index('Model')
results_df

## Model comparison

In [ ]:
results_df['Test accuracy'].plot(kind='bar', ylim=(0, 1), title='Test accuracy by model')
plt.ylabel('Accuracy'); plt.xticks(rotation=20); plt.show()
best_name = results_df['Test accuracy'].idxmax()
print('Best model:', best_name)

## Confusion matrix of every model

In [ ]:
for name, gs in fitted.items():
    plot_confusion_matrix(Y_test, gs.predict(X_test), f'{name} - confusion matrix')

## Conclusion (fill in from your own output)
State the best model, its test accuracy and what the confusion matrix shows (e.g. false positives = predicted landing but did not land). Note that with ~18 test samples, several models often tie.